# FreightLake — MongoDB EDA

Exploratory analysis of the semi-structured logistics event source before Bronze ingestion.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()
client = MongoClient(os.getenv("MONGO_URI", "mongodb://localhost:27017"))
db = client["freight_lake"]

print("Connected to MongoDB:", db.name)

## 1. Collections

In [ ]:
collections = db.list_collection_names()
collections

## 2. Document Counts

In [ ]:
counts = pd.DataFrame(
    [
        {"collection": c, "document_count": db[c].count_documents({})}
        for c in collections
    ]
)
counts.sort_values("document_count", ascending=False)

## 3. Sample Documents

In [ ]:
from pprint import pprint

for collection in collections:
    print(f"\n### {collection}")
    doc = db[collection].find_one({}, {"_id": 0})
    pprint(doc)

## 4. Field Coverage

In [ ]:
from collections import Counter

for collection in collections:
    counter = Counter()
    for doc in db[collection].find({}, {"_id": 0}):
        counter.update(doc.keys())
    total = db[collection].count_documents({})
    result = pd.DataFrame(
        [
            {
                "field": field,
                "document_count": n,
                "coverage_pct": round(n / total * 100, 2),
            }
            for field, n in counter.items()
        ]
    ).sort_values("coverage_pct")
    print(f"\n### {collection}")
    display(result)

## 5. `event_ts` Analysis

In [ ]:
for collection in collections:
    pipeline = [
        {"$match": {"event_ts": {"$exists": True}}},
        {
            "$group": {
                "_id": None,
                "min_event_ts": {"$min": "$event_ts"},
                "max_event_ts": {"$max": "$event_ts"},
                "count": {"$sum": 1},
            }
        },
    ]
    result = list(db[collection].aggregate(pipeline))
    print(f"\n{collection}:")
    pprint(result)

## 6. Variable / Nested Fields

In [ ]:
def flatten_keys(doc, prefix=""):
    keys = []
    for key, value in doc.items():
        path = f"{prefix}.{key}" if prefix else key
        keys.append(path)
        if isinstance(value, dict):
            keys.extend(flatten_keys(value, path))
    return keys


for collection in collections:
    counter = Counter()
    for doc in db[collection].find({}, {"_id": 0}):
        counter.update(flatten_keys(doc))
    print(f"\n### {collection}")
    display(
        pd.DataFrame(
            counter.items(), columns=["field_path", "document_count"]
        ).sort_values("document_count")
    )